In [2]:
import torch
import numpy as np


print(f"PyTorch version: {torch.__version__}")

# Check PyTorch has access to MPS (Metal Performance Shader, Apple's GPU architecture)
print(f"Is MPS (Metal Performance Shader) built? {torch.backends.mps.is_built()}")
print(f"Is MPS available? {torch.backends.mps.is_available()}")

# Set the device      
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

PyTorch version: 2.4.0
Is MPS (Metal Performance Shader) built? True
Is MPS available? True
Using device: mps


In [3]:
import torch
import torch.nn as nn
import time

# Define devices
device_cpu = torch.device("cpu")
device_gpu = torch.device("mps") if torch.backends.mps.is_available() else None

# Define a simple model
class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        return self.net(x)

# Benchmarking function
def benchmark(device, label):
    model = SimpleMLP().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()

    # Dummy data
    x = torch.randn(256, 2048).to(device)
    y = torch.randint(0, 10, (256,)).to(device)

    # Warm-up
    for _ in range(10):
        optimizer.zero_grad()
        output = model(x)
        loss = loss_fn(output, y)
        loss.backward()
        optimizer.step()

    # Timed training
    start = time.time()
    for _ in range(100):
        optimizer.zero_grad()
        output = model(x)
        loss = loss_fn(output, y)
        loss.backward()
        optimizer.step()
    end = time.time()

    print(f"{label} time: {end - start:.4f} seconds")

# Run on CPU
benchmark(device_cpu, "CPU")

# Run on GPU (MPS) if available
if device_gpu:
    benchmark(device_gpu, "GPU (MPS)")
else:
    print("MPS backend not available.")

CPU time: 0.4347 seconds
GPU (MPS) time: 0.1860 seconds


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import time

# Set devices
device_cpu = torch.device("cpu")
device_gpu = torch.device("mps") if torch.backends.mps.is_available() else None

# Synthetic dataset
class SyntheticDataset(Dataset):
    def __init__(self, num_samples=10000, input_dim=2048, num_classes=10):
        self.x = torch.randn(num_samples, input_dim)
        self.y = torch.randint(0, num_classes, (num_samples,))
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

# Simple MLP model
class SimpleMLP(nn.Module):
    def __init__(self, input_dim=2048, num_classes=10):
        super(SimpleMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# Training loop
def train(device, batch_size, label):
    dataset = SyntheticDataset()
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = SimpleMLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    # Warm-up
    for _ in range(1):
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            output = model(x)
            loss = loss_fn(output, y)
            loss.backward()
            optimizer.step()

    # Timed training
    start = time.time()
    for epoch in range(5):  # More epochs = more time
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            output = model(x)
            loss = loss_fn(output, y)
            loss.backward()
            optimizer.step()
    end = time.time()

    print(f"{label} | Batch size {batch_size} | Time: {end - start:.2f} seconds")

# Run benchmarks
for batch_size in [32, 64, 128, 256]:
    train(device_cpu, batch_size, "CPU")
    if device_gpu:
        train(device_gpu, batch_size, "GPU (MPS)")
    else:
        print("MPS backend not available.")

CPU | Batch size 32 | Time: 7.34 seconds
GPU (MPS) | Batch size 32 | Time: 8.80 seconds
CPU | Batch size 64 | Time: 3.95 seconds
GPU (MPS) | Batch size 64 | Time: 3.57 seconds
CPU | Batch size 128 | Time: 2.37 seconds
GPU (MPS) | Batch size 128 | Time: 1.80 seconds
CPU | Batch size 256 | Time: 1.46 seconds
GPU (MPS) | Batch size 256 | Time: 0.90 seconds
